# Batch normalization

_Deep Learning_

**Rescale each layer's inputs to be tidy and centered, so training is faster and smoother.**

As data flows through layers, its scale can drift, getting too big or too small. That slows training.

     Batch normalization cleans up a layer's inputs: it centers them at 0 and scales them to a steady size.

---

This notebook is a practice scaffold for the **Batch normalization** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(16, 32),
    nn.BatchNorm1d(32),       # normalize the 32 features across the batch
    nn.ReLU(),
    nn.Linear(32, 1),
)

x = torch.randn(64, 16) * 5 + 10   # messy inputs: large mean, big spread

net.train()
out = net[1](net[0](x))            # output of the BatchNorm layer
print("after BN mean:", round(out.mean().item(), 3))   # ~ 0
print("after BN std: ", round(out.std().item(), 3))    # ~ 1

## Visualize the data & results

_On real tumor data, does normalizing the inputs (batch-norm's job) let the network train steadily?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss

bc = load_breast_cancer()
Xn = StandardScaler().fit_transform(bc.data)
Xtr_r, Xte_r, ytr, yte = train_test_split(bc.data, bc.target, test_size=0.3,
                                          random_state=0, stratify=bc.target)
Xtr_s, Xte_s, _, _ = train_test_split(Xn, bc.target, test_size=0.3,
                                      random_state=0, stratify=bc.target)

def val_curve(Xt, Xv):
    m = MLPClassifier(hidden_layer_sizes=(64, 64), solver="adam", max_iter=1,
                      warm_start=True, random_state=0, alpha=1e-4,
                      learning_rate_init=0.003)
    va = []
    for e in range(40):
        m.partial_fit(Xt, ytr, classes=[0, 1]) if e == 0 else m.partial_fit(Xt, ytr)
        va.append(log_loss(yte, m.predict_proba(Xv), labels=[0, 1]))
    return va

plt.plot(val_curve(Xtr_r, Xte_r), color="#ff7b72", label="raw inputs (unscaled)")
plt.plot(val_curve(Xtr_s, Xte_s), color="#7ee787", label="normalized inputs")
plt.title("Real validation loss: raw vs normalized breast-cancer inputs")
plt.xlabel("epoch"); plt.ylabel("log loss")
plt.legend()
plt.show()